# 00 — Data Sanity Checks

**Purpose:** verify that the raw data pipeline in `src/data.py` is correct *before* any training happens.

**Why this notebook exists:** roughly 80% of "the model isn't learning" bugs trace back to the data layer — a chromosome-name mismatch, an off-by-one in window extraction, or a leaky train/test split. Those bugs are silent: the model trains, the loss goes down, and the AUROC looks plausible, but the model has learned the wrong thing. This notebook is the checklist that catches them.

**Prerequisites:**
- `data/raw/creb1_peaks.bed` (ENCODE narrowPeak, GRCh38)
- `data/raw/Homo_sapiens.GRCh38.dna.primary_assembly.fa` (Ensembl GRCh38)

If either is missing, run `bash scripts/download_data.sh` from the repo root first.

**When to run:** after any change to `src/data.py`, and once on a fresh clone.

In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import numpy as np
import pandas as pd
from pyfaidx import Fasta

from src.data import (
    CANONICAL_CHROMS, DataConfig, build_dataset,
    load_peaks, one_hot, extract_window, _peak_center,
)

## 1. The BED file looks sane

We expect a narrowPeak file with 10 columns, coordinates in GRCh38, and chromosome names like `chr1` (which `load_peaks` normalizes to `1`).

**What to verify in the output below:**
- `chrom` column is `object` (string), not `int64`
- Only canonical chromosomes appear (`1`–`22`, `X`, `Y`) — no `chrUn_*`, no alt contigs, no `MT`
- `summit >= peak length` is **0** (a nonzero value means the summit falls outside the peak interval, which is malformed narrowPeak)

In [2]:
import os
from pathlib import Path

print("cwd:", os.getcwd())
print("data/raw exists here?", Path("data/raw/creb1_peaks.bed").exists())
print("repo-root-relative exists?", Path("../data/raw/creb1_peaks.bed").exists())

cwd: /home/lux/Documents/code/personal/TFBS-creb1-predicter/notebooks
data/raw exists here? False
repo-root-relative exists? True


In [3]:
cfg = DataConfig()
peaks = load_peaks(cfg.bed_path)

print(f"peaks: {len(peaks):,}")
print(f"columns: {list(peaks.columns)}")
print(f"chrom dtype: {peaks.chrom.dtype}")
print(f"start dtype: {peaks.start.dtype}")
display(peaks.head())
print("\nper-chromosome counts:")
display(peaks.chrom.value_counts().sort_index())

print(f"\nsummit == -1 rows: {(peaks.summit == -1).sum():,}")
print(f"summit >= peak length (bad): {(peaks.summit >= (peaks.end - peaks.start)).sum():,}")

FileNotFoundError: Peak file not found: data/raw/creb1_peaks.bed